# BuildZoom Contractor Extractor (DC, MD, Northern VA)

This notebook collects contractor listings from BuildZoom for Washington DC, nearby Maryland cities, and Northern Virginia. It uses a requests-based scraper by default, with an optional Selenium fallback if pages are heavily scripted.

Notes:
- Respect target site Terms of Service and robots.txt. Use modest concurrency and rate limits.
- Selectors may change; adjust constants in the parser section if needed.
- Output: CSV and Parquet files under `data/`.



## Optional: Enable Selenium rendering
Set `USE_SELENIUM = True` and run the next cell to fetch pages via a headless browser if requests-based parsing misses content. Requires Google Chrome and will auto-manage the driver.


In [3]:
# If you need Selenium, set USE_SELENIUM = True above, then run this cell.
# Make this cell robust to execution order by defaulting when undefined
try:
    USE_SELENIUM
except NameError:
    USE_SELENIUM = False

if USE_SELENIUM:
    # !pip install selenium webdriver-manager
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager

    def make_driver():
        opts = Options()
        opts.add_argument("--headless=new")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--window-size=1280,1200")
        return webdriver.Chrome(ChromeDriverManager().install(), options=opts)

    _driver = make_driver()

    def fetch_html(url: str) -> Optional[str]:  # type: ignore[override]
        try:
            _driver.get(url)
            time.sleep(2.0)
            return _driver.page_source
        except Exception:
            return None

    print("Selenium is enabled for fetching HTML.")
else:
    print("Selenium is disabled. Using requests.")



Selenium is disabled. Using requests.


In [4]:
# If running in a fresh environment, uncomment to install deps
# !pip install requests beautifulsoup4 pandas tqdm fake-useragent lxml selenium webdriver-manager pyarrow

import os
os.makedirs('data', exist_ok=True)
print('Data directory ready at ./data')



Data directory ready at ./data


In [5]:
from __future__ import annotations
import re
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Optional

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

try:
    from fake_useragent import UserAgent
    _ua = UserAgent()
    def get_user_agent() -> str:
        return _ua.random
except Exception:
    def get_user_agent() -> str:
        return (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )

SESSION = requests.Session()
SESSION.headers.update({
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "User-Agent": get_user_agent(),
})

REQUEST_TIMEOUT_S = 20
BASE_URL = "https://www.buildzoom.com"

USE_SELENIUM = False  # Set True if the requests parser misses content
RATE_LIMIT_SECONDS = (1.5, 3.0)  # min/max sleep between requests
MAX_PAGES_PER_CITY = 5  # adjust as needed; set None for all pages



In [6]:
# Region seed pages
REGION_SEEDS = {
    # Washington DC
    "Washington, DC": f"{BASE_URL}/washington-dc/general-contractors",
    # Maryland close to DC
    "Bethesda, MD": f"{BASE_URL}/bethesda-md/general-contractors",
    "Silver Spring, MD": f"{BASE_URL}/silver-spring-md/general-contractors",
    "Rockville, MD": f"{BASE_URL}/rockville-md/general-contractors",
    "Gaithersburg, MD": f"{BASE_URL}/gaithersburg-md/general-contractors",
    "Chevy Chase, MD": f"{BASE_URL}/chevy-chase-md/general-contractors",
    "Potomac, MD": f"{BASE_URL}/potomac-md/general-contractors",
    "Takoma Park, MD": f"{BASE_URL}/takoma-park-md/general-contractors",
    # Northern Virginia
    "Arlington, VA": f"{BASE_URL}/arlington-va/general-contractors",
    "Alexandria, VA": f"{BASE_URL}/alexandria-va/general-contractors",
    "Fairfax, VA": f"{BASE_URL}/fairfax-va/general-contractors",
    "McLean, VA": f"{BASE_URL}/mclean-va/general-contractors",
    "Reston, VA": f"{BASE_URL}/reston-va/general-contractors",
    "Tysons, VA": f"{BASE_URL}/tysons-va/general-contractors",
    "Falls Church, VA": f"{BASE_URL}/falls-church-va/general-contractors",
    "Leesburg, VA": f"{BASE_URL}/leesburg-va/general-contractors",
    "Ashburn, VA": f"{BASE_URL}/ashburn-va/general-contractors",
}
len(REGION_SEEDS)


17

In [7]:
def sleep_a_bit():
    time.sleep(random.uniform(*RATE_LIMIT_SECONDS))


def fetch_html(url: str) -> Optional[str]:
    try:
        SESSION.headers["User-Agent"] = get_user_agent()
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT_S)
        if resp.status_code >= 400:
            return None
        return resp.text
    except requests.RequestException:
        return None


def to_soup(html: Optional[str]) -> Optional[BeautifulSoup]:
    if not html:
        return None
    return BeautifulSoup(html, "lxml")


def extract_text(node) -> str:
    return re.sub(r"\s+", " ", node.get_text(strip=True)) if node else ""


# Heuristics for list-page contractor cards
CARD_SELECTORS = [
    "div.contractor-card",
    "div.search-result",
    "div[data-testid='contractor-card']",
]

NAME_SELECTORS = [
    "a.card-title",
    "a.contractor-name",
    "h2 a",
]

META_SELECTORS = {
    "location": [".location", ".card-subtitle", "[data-testid='contractor-location']"],
    "rating": [".rating", "[data-testid='contractor-rating']", ".bz-rating"],
    "phone": ["a[href^='tel:']", ".phone", "[data-testid='contractor-phone']"],
}

PROFILE_LINK_PATTERN = re.compile(r"^/[^/].*")


def parse_list_page(soup: BeautifulSoup) -> List[Dict]:
    results: List[Dict] = []

    cards = []
    for sel in CARD_SELECTORS:
        cand = soup.select(sel)
        if cand:
            cards = cand
            break
    if not cards:
        # fallback: try common anchors that look like profiles
        for a in soup.select("a[href]"):
            href = a.get("href", "")
            if href.startswith("/") and not href.startswith("/search") and len(href) > 1:
                parent = a.find_parent(["div", "li", "article"]) or a
                cards.append(parent)

    for card in cards:
        # name and profile url
        name = ""
        href = None
        for sel in NAME_SELECTORS:
            node = card.select_one(sel)
            if node:
                name = extract_text(node)
                href = node.get("href")
                break
        if not href:
            a = card.find("a", href=True)
            if a:
                href = a.get("href")
                name = name or extract_text(a)
        if href and href.startswith("/"):
            profile_url = BASE_URL + href
        elif href and href.startswith("http"):
            profile_url = href
        else:
            profile_url = None

        # meta fields
        location = ""
        rating = ""
        phone = ""
        for sel in META_SELECTORS["location"]:
            node = card.select_one(sel)
            if node:
                location = extract_text(node)
                break
        for sel in META_SELECTORS["rating"]:
            node = card.select_one(sel)
            if node:
                rating = extract_text(node)
                break
        for sel in META_SELECTORS["phone"]:
            node = card.select_one(sel)
            if node:
                phone = extract_text(node)
                break

        if name or profile_url:
            results.append({
                "name": name,
                "profile_url": profile_url,
                "location": location,
                "rating": rating,
                "phone": phone,
            })

    return results


def find_next_page_url(soup: BeautifulSoup, current_url: str) -> Optional[str]:
    # Look for next pagination link
    next_link = soup.find("a", string=re.compile(r"^\s*Next\s*$", re.I))
    if not next_link:
        # Look for rel="next"
        next_link = soup.find("a", rel=lambda v: v and "next" in v)
    if next_link and next_link.get("href"):
        href = next_link["href"]
        if href.startswith("http"):
            return href
        if href.startswith("/"):
            return BASE_URL + href
        # join relative
        from urllib.parse import urljoin
        return urljoin(current_url, href)

    # Fallback: try page=? pattern
    from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
    parts = list(urlparse(current_url))
    qs = parse_qs(parts[4])
    page = int(qs.get("page", [1])[0])
    qs["page"] = [str(page + 1)]
    parts[4] = urlencode(qs, doseq=True)
    candidate = urlunparse(parts)
    return candidate if candidate != current_url else None



In [ ]:
EXPECTED_COLUMNS = [
    "name",
    "profile_url",
    "location",
    "rating",
    "phone",
    "region",
    "source_page",
]


def crawl_city(name: str, url: str, max_pages: Optional[int] = MAX_PAGES_PER_CITY) -> pd.DataFrame:
    page_url = url
    page_idx = 1
    rows: List[Dict] = []

    while page_url and (max_pages is None or page_idx <= max_pages):
        html = fetch_html(page_url)
        soup = to_soup(html)
        items = parse_list_page(soup) if soup else []
        for it in items:
            it["region"] = name
            it["source_page"] = page_url
            rows.append(it)
        # advance
        next_url = find_next_page_url(soup, page_url) if soup else None
        if next_url == page_url:
            next_url = None
        page_url = next_url
        page_idx += 1
        sleep_a_bit()

    df = pd.DataFrame(rows)
    # Ensure all expected columns exist even if empty
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    return df[EXPECTED_COLUMNS] if not df.empty else df


def crawl_all_regions(regions: Dict[str, str]) -> pd.DataFrame:
    frames = []
    for region_name, seed in tqdm(regions.items()):
        try:
            df = crawl_city(region_name, seed)
            frames.append(df)
        except Exception:
            continue
    if not frames:
        return pd.DataFrame(columns=EXPECTED_COLUMNS)

    out = pd.concat(frames, ignore_index=True)

    # Ensure columns exist before cleanup
    for col in EXPECTED_COLUMNS:
        if col not in out.columns:
            out[col] = ""

    # basic cleanup & dedupe
    out["name"] = out["name"].fillna("").str.strip()
    out["profile_url"] = out["profile_url"].fillna("")
    out = out.drop_duplicates(subset=["profile_url", "name"], keep="first")
    return out[EXPECTED_COLUMNS]


def save_outputs(df: pd.DataFrame, stem: str = "buildzoom_contractors") -> None:
    csv_path = os.path.join("data", f"{stem}.csv")
    parquet_path = os.path.join("data", f"{stem}.parquet")
    df.to_csv(csv_path, index=False)
    try:
        df.to_parquet(parquet_path, index=False)
    except Exception:
        pass
    print(f"Saved {len(df)} rows to {csv_path}")



In [9]:
# Crawl a small sample (first few pages per city)
df = crawl_all_regions(REGION_SEEDS)
print(df.shape)
df.head(10)


100%|██████████| 17/17 [00:38<00:00,  2.29s/it]


KeyError: 'name'

In [ ]:
if not df.empty:
    save_outputs(df)
else:
    print("No rows extracted. Consider increasing MAX_PAGES_PER_CITY or enabling Selenium.")

